# Stoic Qwen3.8 27B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** `unsloth/Qwen3.8-27B` — bf16 weights (`Qwen3_5ForConditionalGeneration`, `model_type` `qwen3_5`), quantized to bnb NF4 on the fly by `load_in_4bit=True`. There is no pre-quantized bnb-4bit repo for this model; the first run downloads ~56 GB.

**Architecture note:** Qwen3.8 27B is a *multimodal hybrid*, not the text-only dense model this notebook lineage was originally written for. 64 language layers = 48 gated-delta `linear_attn` + 16 `self_attn`, plus a 27-layer vision tower and a 1-layer MTP head. That shapes several choices below (LoRA scoping, packing, length grouping, GGUF EOS).

**Dataset:** Per-persona datagen JSONL — each persona has its own system prompt and distinctive voice.

**Training Hardware:** NVIDIA DGX Spark (GB10, sm_120, 128 GB unified memory).

**Deployment Target:** The PEFT LoRA adapter is the artifact that ships. Serve it against a compatible Qwen3.8 27B base after the smoke test. The GGUF export cells at the end are optional.

**Chat Template:** Tokenizer-native Qwen ChatML (`<|im_start|>role\ncontent<|im_end|>`), applied via `tokenizer.apply_chat_template`.

**Thinking mode:** Qwen3.8 thinks by default. The SFT formatting cell explicitly passes `enable_thinking=False` so the template does not inject Qwen3.8's default xhigh reasoning instruction into every training example. `ENABLE_THINKING` in Cell 1 controls only the inference/eval cells.

**Architecture:** This base LoRA teaches persona-switching — "when the system prompt says you're Marcus Aurelius, speak like Marcus; when it says Seneca, speak like Seneca." Optional persona LoRAs can refine individual voices further.

**Lineage:** Recipe verified against `biblical/notebooks/loras/qwen_38_27b/biblical_qwen3_8_27b_sft_unsloth_4bit.ipynb`, which produced working adapters on this machine against this exact base. Its DPO counterpart here is `stoic_qwen38_27b_dpo.ipynb`.

## 1. Configuration

All paths and variables for easy configuration.

In [ ]:
import os

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
# Normally run inside the `unsloth-notebook` container, which bind-mounts
# /home/spark/projects/training -> /workspace/training. The fallbacks let the same
# notebook run on the host without edits.
if os.path.exists("/workspace/training/stoic"):
    PROJECT_ROOT = "/workspace/training/stoic"
elif os.path.exists("/workspace/stoic"):
    PROJECT_ROOT = "/workspace/stoic"
else:
    PROJECT_ROOT = "/home/spark/projects/training/stoic"

OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== HUGGING FACE CACHE ===========================
# Use the default Hub cache: /root/.cache/huggingface/hub in the unsloth-notebook container,
# which is bind-mounted from /home/spark/.cache/huggingface/hub on the host. Do NOT override
# HF_HUB_CACHE: pointing it at the cache root (not hub/) made unsloth build a second
# models--* tree beside hub/ and re-download every base model.

# =========================== MODEL CONFIGURATION ===========================
# unsloth/Qwen3.8-27B ships bf16 weights (~56 GB, Qwen3_5ForConditionalGeneration,
# model_type qwen3_5). No Unsloth pre-quantized bnb-4bit repo exists for Qwen3.8-27B,
# so Unsloth quantizes to bnb NF4 on the fly via load_in_4bit=True in the model cell.
BASE_LLM = "unsloth/Qwen3.8-27B"

# NOTE: stoic_qwen38_27b_dpo.ipynb reads this exact string as SFT_MODEL_NAME_BASE to
# locate the adapter it continues from. Change it in both places or not at all.
MODEL_NAME_BASE = "stoic_qwen38_27b_unsloth_4bit"

# =========================== THINKING MODE ===========================
# Qwen3.8's chat template accepts `enable_thinking`. It affects ONLY calls made with
# add_generation_prompt=True (i.e. inference):
#   False -> '<|im_start|>assistant\n<think>\n\n</think>\n\n'  (empty block; answer directly)
#   True  -> '<|im_start|>assistant\n<think>\n'                   (model opens a reasoning block)
# It also gates whether the template prepends its default xhigh reasoning instruction to
# the system prompt. The training formatting cell ALWAYS passes enable_thinking=False so
# that instruction never contaminates the persona system prompts; this flag controls only
# the inference/eval cells, which must match training to be a meaningful smoke test.
ENABLE_THINKING = False

# =========================== INPUT DATA ===========================
# Combined multi-turn ShareGPT JSONL (SFT) from the datagen notebooks.
# Already quality-filtered, multi-turn (up to 4 QA pairs per conversation), grouped by topic.
INPUT_DATA_FILE = f"{PROJECT_ROOT}/data/training-data/stoic_persona/stoic_personas_combined_sharegpt.jsonl"

# =========================== PERSONA SYSTEM PROMPTS ===========================
# System prompts are EXTRACTED from the JSONL at load time (see the data loading cell).
# This keeps training in sync with datagen — regenerate data with new/changed prompts and
# this notebook picks them up automatically. After loading, `persona_system_prompts` maps
# persona_key -> full prompt text, and is saved alongside the LoRA adapters for inference.

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
# Measured with the Qwen3.8 tokenizer over all 25,232 conversations rendered through the
# chat template: p50=1666, p90=2211, p99=2461, max=3097 tokens. 4096 truncates nothing.
# Packing is off and batches pad to their own longest member, so the headroom is free.
MAX_SEQ_LENGTH = 4096
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
TARGET_EPOCHS = 1
SFT_MAX_EXAMPLES = 3000  # 0 = use all valid conversations; set e.g. 500/1000/3000 for capped runs

# =========================== CHECKPOINTING ===========================
# A capped run is a few hundred steps and still runs for hours. Checkpoint often enough
# that a crash costs minutes; the training cell auto-resumes from the newest checkpoint.
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 3

# gc.collect() + torch.cuda.empty_cache() cadence during training. Aligned with the
# checkpoint interval — right after a save is when transient buffers are safe to drop.
CLEANUP_STEPS = 50

# =========================== LoRA CONFIGURATION ===========================
# Adapter is merged into the base weights at GGUF export, so adapter size on disk is
# irrelevant. Higher rank for stronger persona distinction.
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0

# These leaf names are the standard attention + MLP projections. On Qwen3.8 they resolve to
# 192 language MLP modules (all 64 layers) + 64 language self_attn modules (the 16
# full-attention layers only). The 48 gated-delta `linear_attn` layers use different leaf
# names (in_proj_qkv / in_proj_a / in_proj_b / in_proj_z / out_proj) and are deliberately
# NOT adapted — this matches the confirmed-working Biblical Qwen3.8 adapter, whose
# saved adapter_config resolves to exactly these 256 modules.
#
# IMPORTANT: the LoRA cell pairs this list with finetune_vision_layers=False. Without that
# flag Unsloth passes the bare list to PEFT, which matches by SUFFIX across the whole
# model and also adapts the 7 `mtp.*` modules — a head that receives no gradient in a
# causal-LM forward and that vLLM rejects in an adapter. See that cell for the guard.
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
# Defaults taken from the base model's own generation_config.json.
TEST_PROMPT = "I am struggling with anxiety about things outside my control. How do I find peace?"
GEN_TEMPERATURE = 1.0
GEN_TOP_P = 0.95
GEN_TOP_K = 20

# ============================================================================
print("Configuration loaded (Qwen3.8 27B 4-bit QLoRA, v2 SFT)")
print(f"  Project root:     {PROJECT_ROOT}")
print(f"  HF hub cache:     {os.environ.get('HF_HUB_CACHE', '<default>')}")
print(f"  Base model:       {BASE_LLM}")
print(f"  Model name:       {MODEL_NAME_BASE}")
print(f"  Input data:       {INPUT_DATA_FILE}")
print(f"  Output base:      {OUTPUT_BASE_DIR}")
print(f"  LoRA output:      {LORA_OUTPUT_DIR}")
print(f"  LoRA config:      r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training:         batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM} "
      f"(effective {BATCH_SIZE * GRAD_ACCUM}), lr={LEARNING_RATE}, epochs={TARGET_EPOCHS}")
print(f"  SFT max examples: {SFT_MAX_EXAMPLES or 'ALL'}")
print(f"  Max seq length:   {MAX_SEQ_LENGTH} (dataset max is 3097 tokens)")
print(f"  Checkpoints:      every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Thinking mode:    {'ON' if ENABLE_THINKING else 'OFF'} (inference/eval cells only)")
print(f"  Persona prompts:  extracted from JSONL at load time")

if not os.path.exists(INPUT_DATA_FILE):
    raise FileNotFoundError(f"Training data not found: {INPUT_DATA_FILE}")

## 2. Environment Preparation

Safe to re-run. Order matters:

1. Verify the CUDA PyTorch build is intact (no side effects).
2. Core training packages, and remove the aarch64 `torchao` build that blocks PEFT's bnb 4-bit dispatcher.
3. `transformers` + `peft` from git main — the `qwen3_5` architecture is newer than the container's release.
4. Small utility packages.
5. Rebuild `causal_conv1d` from source — the NGC image ships the Python package **without** its compiled CUDA extension, which hard-crashes any import that touches the FalconH1 model inside `transformers`/`unsloth`.
6. `flash-linear-attention` — Triton kernels for the Qwen3.8 gated-delta-rule fast path.
7. Import `unsloth` **before** `transformers` so its monkey-patches apply.

**Restart the kernel after this cell, then resume from Cell 1.**

In [ ]:
import os, sys, subprocess, importlib, importlib.util

def _pip(*args, env_extra=None):
    """Run pip against this kernel's interpreter; print output only on failure."""
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args], capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# --- 1. Verify the CUDA PyTorch build is intact ------------------------------
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "cpu" in torch.__version__:
        print("  CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")
print(f"  torch {torch.__version__} - CUDA {torch.version.cuda} - GPU: {torch.cuda.get_device_name(0)}")

# --- 2. Core training packages ------------------------------------------------
print("  Installing core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...")
_pip("install", "-q", "-U", "unsloth", "trl", "accelerate", "datasets", "bitsandbytes")

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel >=0.16 on PyPI, so
# uninstall - peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
_pip("uninstall", "-y", "-q", "torchao")

# --- 3. transformers + peft from git main -------------------------------------
# Qwen3.8 uses the `qwen3_5` architecture, which is newer than the transformers
# release the container ships. Install from git main so the arch is recognised.
print("  Installing transformers + peft from git main...")
_pip("install", "-q", "-U", "git+https://github.com/huggingface/transformers.git")
_pip("install", "-q", "-U", "git+https://github.com/huggingface/peft.git")

# --- 4. Small utility packages ------------------------------------------------
for _module, _install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(_module) is None:
        print(f"  Installing {_install_args[-1]}...")
        _pip(*_install_args)

# --- 5. Fix causal_conv1d -----------------------------------------------------
# The NGC image ships the causal_conv1d Python package WITHOUT its compiled CUDA
# extension (causal_conv1d_cuda). That hard-crashes any import that reaches the
# FalconH1 model inside transformers or unsloth, so it must be fixed BEFORE
# importing either.
#
# pip also caches a broken prebuilt aarch64 wheel, so --no-binary is required to
# force a source build, together with CAUSAL_CONV1D_FORCE_BUILD=TRUE. The first
# build takes a few minutes on aarch64; pip caches the result afterwards.
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",   # DGX Spark GB10 = sm_120
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    # Clean up - don't leave partial imports that spoil unsloth's import order
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing - rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    _ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
               "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if _ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext - uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed - uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# --- 6. flash-linear-attention ------------------------------------------------
# fla provides the Triton JIT kernels (chunk_gated_delta_rule etc.) used by the
# Qwen3.8 fast path. fla-core installs into the same `fla` namespace.
if _check_import("fla") is None:
    print("  Installing flash-linear-attention...")
    _pip("install", "-q", "--no-deps", "flash-linear-attention", "fla-core")

_fast_path_ok = False
try:
    from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gated_delta_rule
    _fast_path_ok = _causal_ok and chunk_gated_delta_rule is not None
    for _k in list(sys.modules.keys()):
        if _k.startswith("fla."):
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError):
    pass
print(f"  Fast path: {'ENABLED' if _fast_path_ok else 'DISABLED (using torch fallback)'}")

# --- 7. Import unsloth FIRST, then transformers -------------------------------
# Unsloth must be imported before transformers/trl/peft so its monkey-patches apply.
# Purge anything already loaded so the imports pick up the versions installed above.
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
import peft
import trl

print()
print(f"  unsloth:       {unsloth.__version__}")
print(f"  transformers:  {transformers.__version__}")
print(f"  peft:          {peft.__version__}")
print(f"  trl:           {trl.__version__}")
print(f"  causal_conv1d: {'OK' if _causal_ok else 'FALLBACK (torch path)'}")
print(f"  fla:           {'OK' if _check_import('fla') else 'MISSING'}")
print(f"  torchao:       {'PRESENT (should be absent)' if importlib.util.find_spec('torchao') else 'absent (correct)'}")
print()
print("Environment ready. Restart kernel, then rerun from Cell 1 (Configuration).")

## 3. Load Dataset

Load the combined multi-turn ShareGPT JSONL from `stoic_datagen.ipynb`.

- Already quality-filtered (no short answers, no AI refusals)
- Multi-turn: QA pairs grouped per conversation by topic
- Each conversation has a persona-specific system prompt
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`

In [ ]:
import json, os, re, random
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING COMBINED SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

# Load multi-turn conversations and EXTRACT system prompts from the JSONL.
# This replaces hardcoded prompt dicts — prompts stay in sync with datagen automatically.
conversations = []
persona_system_prompts = {}   # persona_key -> full system prompt text
persona_counts = defaultdict(int)
persona_by_index = []         # parallel to `conversations`, used for the stratified cap below

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)
        conversations.append(conv)

        # Extract persona name from "You are <Name>, ..." pattern
        sys_msg = conv["conversations"][0]["value"]
        match = re.match(r"You are (.+?),", sys_msg)
        if match:
            raw_name = match.group(1)
            # Normalize to snake_case key: lowercase, strip leading "the ", underscores for spaces
            key = raw_name.lower()
            key = re.sub(r"^the\s+", "", key)
            key = key.replace(" ", "_")
            persona_counts[key] += 1
            if key not in persona_system_prompts:
                persona_system_prompts[key] = sys_msg
        else:
            key = "unknown"
            print(f"  WARNING: could not extract persona from system prompt: {sys_msg[:80]}...")
        persona_by_index.append(key)

dataset = HFDataset.from_list(conversations)

# Apply SFT cap from config — stratified per persona, deterministic seed.
# Applied AFTER loading the JSONL, BEFORE quality validation (matches the PubMed SFT notebook).
#
# Two fixes over the previous revision:
#   1. WATER-FILLING. An even split takes min(target, available) per persona and used to
#      drop the shortfall on the floor. With this corpus that silently turned a requested
#      3000 into 2361 back when Epicurus was in the corpus with only 111 conversations
#      against a 750 target. He has since been removed (data/scripts/remove_persona.py),
#      but the shortfall bug is real for any imbalanced persona mix, so the fix stays.
#      The shortfall is now redistributed over the personas that still have rows left,
#      repeating until the cap is met or every pool is exhausted.
#   2. RANDOM SAMPLING. The old code seeded the RNG and then took pool[:take] — the first
#      N rows, not a sample. The JSONL is grouped by topic, so that biased every capped run
#      toward whichever topics datagen happened to emit first. Now uses random.sample.
if SFT_MAX_EXAMPLES:
    if SFT_MAX_EXAMPLES < 0:
        raise ValueError(f"SFT_MAX_EXAMPLES must be 0 or positive, got {SFT_MAX_EXAMPLES}")
    random.seed(42)
    if len(dataset) > SFT_MAX_EXAMPLES:
        idx_by_persona = defaultdict(list)
        for i, p in enumerate(persona_by_index):
            idx_by_persona[p].append(i)

        # Water-fill: repeatedly split the remaining budget evenly across personas that
        # still have unallocated rows, capping each at what it actually has.
        personas_sorted = sorted(persona_counts.keys(), key=lambda p: -persona_counts[p])
        allocated = {p: 0 for p in personas_sorted}
        remaining = SFT_MAX_EXAMPLES
        open_personas = [p for p in personas_sorted if persona_counts[p] > 0]
        while remaining > 0 and open_personas:
            share = remaining // len(open_personas)
            if share == 0:
                # Fewer rows left than personas: hand out one each, largest pools first.
                for p in open_personas[:remaining]:
                    allocated[p] += 1
                    remaining -= 1
                break
            progressed = False
            for p in list(open_personas):
                take = min(share, persona_counts[p] - allocated[p])
                if take <= 0:
                    open_personas.remove(p)
                    continue
                allocated[p] += take
                remaining -= take
                progressed = True
                if allocated[p] >= persona_counts[p]:
                    open_personas.remove(p)
            if not progressed:
                break

        selected_idx = []
        for p in personas_sorted:
            pool = idx_by_persona[p]
            selected_idx.extend(random.sample(pool, allocated[p]))
        random.shuffle(selected_idx)

        even_target = SFT_MAX_EXAMPLES // len(personas_sorted)
        print(f"  Stratified cap: target {SFT_MAX_EXAMPLES} across {len(personas_sorted)} personas "
              f"(even split would be {even_target}/persona; shortfalls redistributed)")
        print(f"  Actually selected: {len(selected_idx)} rows")
        for p in personas_sorted:
            flag = "  <- pool exhausted" if allocated[p] >= persona_counts[p] else ""
            print(f"    {p:<25} {allocated[p]:>5} / {persona_counts[p]:<6} available{flag}")
        if len(selected_idx) < SFT_MAX_EXAMPLES:
            print(f"  NOTE: only {len(selected_idx)} rows available in total (< {SFT_MAX_EXAMPLES})")

        dataset = dataset.select(selected_idx)
        persona_counts = defaultdict(int, {p: allocated[p] for p in personas_sorted if allocated[p]})
    else:
        print(f"  SFT_MAX_EXAMPLES={SFT_MAX_EXAMPLES}, but only {len(dataset)} examples available; using all")

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} multi-turn conversations across {len(persona_counts)} personas")
print(f"Extracted {len(persona_system_prompts)} unique system prompts from JSONL")
print(f"Columns: {dataset.column_names}")
print(f"\nPer-persona breakdown:")
for p, c in sorted(persona_counts.items(), key=lambda x: -x[1]):
    print(f"  {p:20s} {c:>5d} conversations")

# Flag severe imbalance — a persona with a tiny share will not hold its voice after training.
_total = sum(persona_counts.values())
_weak = [(p, c) for p, c in persona_counts.items() if c / _total < 0.05]
if _weak:
    print(f"\n  WARNING: personas under 5% of the training mix — expect weak voice separation:")
    for p, c in sorted(_weak, key=lambda x: x[1]):
        print(f"    {p:<25} {c:>5} ({100 * c / _total:.1f}%)")

# Show a sample prompt to verify extraction
sample_key = next(iter(persona_system_prompts))
print(f"\n--- Sample extracted prompt ({sample_key}, first 200 chars) ---")
print(f"  {persona_system_prompts[sample_key][:200]}...")

## 4. Validate & Summarize Dataset

Datagen data is already clean (no artifacts to strip). Verify data quality and show persona distribution.

In [ ]:
bad_examples = []
empty_responses = []
unique_system_prompts = set()

for i, example in enumerate(dataset):
    convs = example["conversations"]
    # Multi-turn ShareGPT: system, then alternating human/gpt pairs
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count >=3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    # Validate alternating human/gpt after system
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    # Check last GPT response is not empty
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    unique_system_prompts.add(convs[0]["value"])

# Turn-count distribution
from collections import Counter
turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  Unique system prompts: {len(unique_system_prompts)} (should match extracted count: {len(persona_system_prompts)})")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\nBad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_responses:
    print(f"\nFiltering {len(empty_responses)} empty responses...")
    _drop = set(empty_responses)
    # Keep persona_counts consistent with the rows that actually survive.
    for i in _drop:
        s = dataset[i]["conversations"][0]["value"]
        m = re.match(r"You are (.+?),", s)
        k = re.sub(r"^the\s+", "", m.group(1).lower()).replace(" ", "_") if m else "unknown"
        if persona_counts.get(k):
            persona_counts[k] -= 1
    good_indices = [i for i in range(len(dataset)) if i not in _drop]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

# Persona distribution
print(f"\nPERSONA DISTRIBUTION:")
max_name_len = max(len(n) for n in persona_counts)
for name, count in sorted(persona_counts.items(), key=lambda x: -x[1]):
    bar = "#" * (count // 25)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(persona_counts.values()):>5}")

# Show voice differentiation — first response from different personas
print(f"\nVOICE SAMPLES (first ~100 chars of response):")
seen_personas = set()
for example in dataset:
    system = example["conversations"][0]["value"]
    name_part = system.split(",")[0].replace("You are ", "")
    if name_part not in seen_personas and len(seen_personas) < 4:
        response_start = example["conversations"][2]["value"][:100]
        print(f"  {name_part}: \"{response_start}...\"")
        seen_personas.add(name_part)

print(f"\nDataset validated and ready for training")

## 5. Load Model & Tokenizer (4-bit)

Loads `unsloth/Qwen3.8-27B` bf16 weights and quantizes to bnb NF4 on the fly.

**First run downloads ~56 GB** into the shared HF cache configured in Cell 1. Subsequent runs reuse it.

In [ ]:
import os
# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and the broken
# torch.compile path. Carried over from the working Gemma 4 12B and Biblical Qwen3.8
# notebooks on this machine. Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel
import torch

# NOTE: no torch.cuda.set_per_process_memory_fraction() here, deliberately. On GB10 host
# and device share one 128 GB pool, so a fractional cap does not protect anything — it just
# adds a ceiling the bf16 GGUF merge later in this notebook (~56 GB) can trip over. The
# confirmed-working Biblical Qwen3.8 run does not set it either.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Qwen3.8 27B is Qwen3_5ForConditionalGeneration, so Unsloth returns a multimodal
# Processor, not a raw tokenizer. Unwrap it and work with the inner tokenizer for the
# rest of the notebook — the same pattern the confirmed PubMed Qwen3.6 27B SFT notebooks
# use (pubmed/notebooks/loras/qwen36/). This is what actually keeps the vision half of
# the model out of the training path:
#   - SFTTrainer gets a real PreTrainedTokenizer, so TRL does not route through its
#     vision/process_row() pipeline (the thing the DPO notebooks patch around with
#     trainer.is_vision_model = False).
#   - Trainer._get_train_sampler reads processing_class.model_input_names[0]. On a
#     Processor that is "pixel_values", which breaks length grouping; on the tokenizer
#     it is "input_ids".
#   - No image/video preprocessing path is reachable from a text-only batch.
# `processor` is kept only so the adapter directory is saved with the same files the
# confirmed Biblical Qwen3.8 adapter carries.
processor = None
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from Processor — text-only SFT mode)")

# Pad token: set pad = eos for causal LM training.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
if getattr(model, "generation_config", None) is not None:
    model.generation_config.pad_token_id = tokenizer.pad_token_id

vocab_size = getattr(tokenizer, "vocab_size", "unknown")

print(f"Model loaded: {BASE_LLM}")
print(f"  Precision: 4-bit (QLoRA, quantized on load)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {vocab_size}")
print(f"  Tokenizer class: {type(tokenizer).__name__}"
      f"{' (unwrapped from ' + type(processor).__name__ + ')' if processor else ''}")
print(f"  pad_token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})  "
      f"eos_token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"  Attn impl: {getattr(model.config, '_attn_implementation', 'unknown')}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 6. Format Dataset for Chat Template

Apply Qwen's ChatML template to each conversation and build the final training dataset.

Rendered with `add_generation_prompt=False` and `enable_thinking=False`, so Qwen3.8 does not inject its default xhigh reasoning instruction into every training conversation. Assistant turns still render with an empty `<think>\n\n</think>` block, which is exactly what a non-thinking generation prompt produces at inference — so training and serving agree.

In [ ]:
# Standardize ShareGPT rows, then format with the Qwen tokenizer chat template
from unsloth.chat_templates import standardize_sharegpt

original_dataset_len = len(dataset)

dataset = standardize_sharegpt(dataset)
formatted_texts = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
    enable_thinking=False,   # always False for TRAINING, regardless of ENABLE_THINKING
)

# Build final dataset
import pandas as pd
from datasets import Dataset as HFDataset

dataset = HFDataset.from_pandas(pd.DataFrame({"text": formatted_texts}))

# Filter out empty examples and shuffle. SFT_MAX_EXAMPLES is already applied — the
# stratified per-persona cap runs in the data loading cell, before this formatting step.
dataset = dataset.filter(lambda x: len(x["text"]) > 0)
filtered_dataset_len = len(dataset)
dataset = dataset.shuffle(seed=42)

# Sanity check: the reasoning instruction must NOT be present in training text.
_leaked = sum(1 for t in dataset["text"][:256] if "Reasoning effort is set to" in t)
if _leaked:
    raise RuntimeError(
        f"{_leaked}/256 sampled training examples contain Qwen3.8's default reasoning "
        "instruction. enable_thinking=False did not take effect — do not start training."
    )

print(f"--- Sample formatted text (first 500 chars) ---")
print(dataset[0]["text"][:500])
print(f"\nDataset formatted: {len(dataset)} examples")
print(f"  Original conversations: {original_dataset_len}")
print(f"  After empty filtering:  {filtered_dataset_len}")
print(f"  SFT cap:                {SFT_MAX_EXAMPLES or 'ALL'}")
print(f"  Reasoning-instruction leak check: clean")

## 7. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See Cell 1 config for module options.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # These four flags are load-bearing, not decoration. Unsloth only routes an explicit
    # target_modules list through its family-scoping regex when at least one of them is
    # off. Left at their defaults (all True), the bare leaf-name list goes straight to
    # PEFT, which matches by SUFFIX across the WHOLE model — and Qwen3.8's MTP head
    # exposes the same q/k/v/o/gate/up/down leaf names. That would attach 7 adapters to
    # `mtp.*`, a module that receives no gradient in a causal-LM forward and that vLLM
    # rejects when loading the adapter. finetune_vision_layers=False scopes the match to
    # `model.language_model.*`, reproducing the confirmed-working Biblical Qwen3.8
    # adapter exactly: 192 language MLP + 64 language self_attn = 256 modules.
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    # Deliberately True, NOT "unsloth". The "unsloth" path copies saved activations into
    # pinned CPU buffers that only ever grow - unsloth_zoo/gradient_checkpointing.py
    # resizes them up and never down. On GB10 host and device share one 128 GB pool, so
    # that offload frees no capacity while ratcheting unreclaimable pinned pages upward
    # for hours. True uses standard recompute checkpointing with no host copies.
    use_gradient_checkpointing=True,
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

# ============ VERIFY THE ADAPTER SCOPE AND THE VISION/MTP FREEZE ============
# Two separate things are checked here, because they fail in different ways:
#   1. WHERE ADAPTERS ATTACHED. If the finetune_* flags above did not take effect, the
#      bare leaf-name list reaches PEFT and matches by suffix across the whole model,
#      adapting the 7 `mtp.*` modules. vLLM rejects an adapter carrying those keys.
#   2. WHAT IS TRAINABLE. Under QLoRA the base weights are frozen 4-bit and only LoRA
#      A/B carry gradients, so the vision tower and MTP head are already frozen by
#      construction - no explicit requires_grad_(False) pass is needed. That is an
#      assumption worth asserting rather than trusting, so we check it directly:
#      nothing under `visual.` or `mtp.` may be trainable when training starts.
from collections import Counter

_adapted = sorted({
    n.split(".lora_A")[0].split(".lora_B")[0].replace("base_model.model.", "")
    for n, _ in model.named_parameters() if ".lora_A." in n or ".lora_B." in n
})
_families = Counter()
for n in _adapted:
    if n.startswith("mtp."):   _families["mtp (SHOULD BE 0)"] += 1
    elif ".visual." in n:      _families["vision (SHOULD BE 0)"] += 1
    elif ".self_attn." in n:   _families["language self_attn"] += 1
    elif ".mlp." in n:         _families["language mlp"] += 1
    else:                      _families[f"other: {n}"] += 1

print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"  Adapted modules: {len(_adapted)}")
for fam, count in sorted(_families.items()):
    print(f"    {fam:<28} {count}")

# --- Check 1: adapter placement ---
_stray = [n for n in _adapted if n.startswith("mtp.") or ".visual." in n]
if _stray:
    raise RuntimeError(
        f"LoRA attached to {len(_stray)} module(s) outside the language model "
        f"(e.g. {_stray[:3]}). The finetune_* scoping flags did not take effect. "
        "Do not start training - the adapter will not load in vLLM."
    )

# --- Check 2: vision tower and MTP head are frozen ---
def _zone_of(param_name):
    """Classify a parameter by which sub-model it belongs to."""
    n = param_name.replace("base_model.model.", "", 1)
    if n.startswith("mtp."):
        return "mtp head"
    if ".visual." in n or n.startswith("visual."):
        return "vision tower"
    return None

_unfrozen = {}
_zone_params = Counter()
for name, param in model.named_parameters():
    zone = _zone_of(name)
    if zone is None:
        continue
    _zone_params[zone] += 1
    if param.requires_grad:
        _unfrozen.setdefault(zone, []).append(name)

for zone in ("vision tower", "mtp head"):
    n_trainable = len(_unfrozen.get(zone, []))
    status = (f"FROZEN ({_zone_params[zone]} params)" if n_trainable == 0
              else f"{n_trainable} of {_zone_params[zone]} params TRAINABLE - BAD")
    print(f"  {zone:<14} {status}")

if _unfrozen:
    _sample = [n for names in _unfrozen.values() for n in names][:5]
    raise RuntimeError(
        f"{sum(len(v) for v in _unfrozen.values())} parameter(s) in the vision tower / MTP "
        f"head are trainable (e.g. {_sample}). This is a text-only fine-tune - those must "
        "stay frozen. Do not start training."
    )

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"  Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
print(f"  Note: the 48 gated-delta linear_attn layers are not adapted by design "
      f"(different leaf names); their MLPs are.")

## 8. Trainer Setup

- 1 epoch at 2e-4 — gently teaches persona-switching without degrading base capabilities
- Each example has a persona-specific system prompt so the model learns distinct voices
- Loss is computed on assistant turns only (prompt tokens masked to -100)

In [ ]:
import gc
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig


class PeriodicMemoryCleanup(TrainerCallback):
    """Collect dead Python objects and return cached CUDA blocks to the allocator.

    Runs every `every` optimizer steps. This does not repair a leak - it releases
    cached/fragmented blocks that would otherwise sit reserved but unused, which
    matters on GB10 where host and device draw from the same 128 GB pool.
    """

    def __init__(self, every=50):
        self.every = max(1, int(every))

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every == 0:
            gc.collect()
            torch.cuda.empty_cache()
        return control


trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        # Explicitly False. Qwen3.8 is a hybrid linear-attention model; its gated-delta
        # recurrent state and causal conv1d leak across sequence boundaries once packing
        # flattens a batch, so Unsloth force-disables packing for it regardless. The
        # previous revision of this notebook set packing=False but PRINTED "Packing:
        # enabled" - a silent lie about what actually ran.
        packing=False,
        # NOTE: train_sampling_strategy="group_by_length" does NOT work with this model
        # and is deliberately absent. Two independent blockers, both verified in
        # transformers source:
        #   1. Trainer._get_train_sampler passes processing_class.model_input_names[0] to
        #      LengthGroupedSampler. Qwen3.8's processing_class is a Processor, so that is
        #      "pixel_values", not "input_ids" -> ValueError before step 0.
        #   2. Supplying a precomputed "length" column does not help: _remove_unused_columns
        #      prunes every column outside the model forward signature, and it runs BEFORE
        #      sampler_fn in _get_dataloader, so the column is gone by sampling time.
        # Default random sampler is used instead; batches still pad to their own longest
        # member, just without length bucketing.
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        dataloader_pin_memory=False,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
    ),
    callbacks=[PeriodicMemoryCleanup(CLEANUP_STEPS)],
)

# ===================== TRAIN ON RESPONSES ONLY =====================
# Mask prompt tokens to -100 so loss is computed ONLY on assistant turns.
# Without this the loss averages in the persona system prompt and the user turns -
# tokens the base model already predicts near-perfectly and that repeat across every
# example. They pin the average near their own ~0 loss and bury the signal from the
# response tokens. The persona system prompts here are ~1,600 characters each and
# repeat across thousands of rows, so this notebook is a worst case for that dilution.
#
# Both marker args are left as None so Unsloth auto-detects them from the Qwen chat
# template and prints what it found - no hand-written marker strings to get wrong.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

# Verify the markers actually matched. If they do not, every label can end up -100,
# which trains on nothing and silently burns the entire run.
import numpy as np

_probe = trainer.train_dataset[:8]["labels"]
_kept = sum(int((np.array(x) != -100).sum()) for x in _probe)
_total = sum(len(x) for x in _probe)
if _kept == 0:
    raise RuntimeError(
        "train_on_responses_only masked EVERY token - the instruction/response "
        "markers did not match the chat template. Do not start training."
    )
print(f"Response masking OK: {_kept:,}/{_total:,} label tokens kept "
      f"({100 * _kept / _total:.1f}%) across 8 sample sequences")

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
print(f"Trainer configured")
print(f"  Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: disabled (hybrid linear-attention model)")
print(f"  processing_class: {type(tokenizer).__name__} (Processor unwrapped at load)")
print(f"  Checkpoints: every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Memory cleanup: every {CLEANUP_STEPS} steps")
print(f"  Loss computed on: assistant responses only (prompt masked to -100)")
print(f"  Dataset: {len(dataset)} examples")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

## 9. Train

In [ ]:
# Start training. Auto-resumes from the newest checkpoint in OUTPUT_DIR_ADAPTERS if one
# exists, so an interrupted run continues instead of restarting from step 0.
import os
from transformers.trainer_utils import get_last_checkpoint

_ckpt_dir = trainer.args.output_dir
last_checkpoint = get_last_checkpoint(_ckpt_dir) if os.path.isdir(_ckpt_dir) else None

if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
    result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found - starting from scratch.")
    result = trainer.train()

print("\nTraining complete")
print(f"  Final loss:  {result.training_loss:.4f}")
print(f"  Total steps: {result.global_step}")

## 10. Save LoRA Adapters

Save the trained LoRA adapters plus the persona system prompts. The adapters load on any quantization of the same Qwen3.8 27B base via PEFT or vLLM.

`stoic_qwen38_27b_dpo.ipynb` continues training from exactly this directory.

In [ ]:
from pathlib import Path
import json

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Save LoRA adapters + tokenizer
print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
# Save the Processor when we unwrapped one, so the adapter directory carries the same
# files the confirmed Biblical Qwen3.8 adapter does (processor_config.json alongside
# tokenizer_config.json / chat_template.jinja). Saving a Processor also writes the
# tokenizer files, so this is a superset of tokenizer.save_pretrained().
(processor or tokenizer).save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

# Machine-readable completion sentinel (docs/README.md: "Final model-producing
# notebooks must write a machine-readable completion sentinel").
from datetime import datetime, timezone
sentinel = {
    "status": "complete",
    "stage": "sft",
    "model_name": MODEL_NAME_BASE,
    "base_model": BASE_LLM,
    "input_data_file": INPUT_DATA_FILE,
    "output_dir": OUTPUT_DIR_ADAPTERS,
    "lora_output_dir": LORA_OUTPUT_DIR,
    "global_step": int(result.global_step),
    "final_loss": float(result.training_loss),
    "num_train_epochs": TARGET_EPOCHS,
    "sft_max_examples": SFT_MAX_EXAMPLES,
    "train_examples": len(dataset),
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "adapted_modules": len(_adapted),
    "personas": sorted(persona_system_prompts),
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
with open(f"{LORA_OUTPUT_DIR}/complete.json", "w") as f:
    json.dump(sentinel, f, indent=2)

example_persona = next(iter(persona_system_prompts), "marcus_aurelius")

print(f"\nLoRA adapters saved")
print(f"  Sentinel:       {LORA_OUTPUT_DIR}/complete.json (step {result.global_step}, "
      f"loss {result.training_loss:.4f})")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(persona_system_prompts)} personas)")
print(f"\n  At inference, load prompts with:")
print(f'    with open("{prompts_path}") as f:')
print(f'        prompts = json.load(f)')
print(f'    system_msg = prompts["{example_persona}"]  # or any persona key')

## 11. Test Inference

Quick smoke test with a few personas using their extracted system prompts. Each persona should respond in its distinctive voice.

Uses `ENABLE_THINKING` from Cell 1 so the rendered prompt matches what training saw. With it `True`, output begins inside a `<think>` block before the answer, so `max_new_tokens` is raised to leave room for both.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Pick up to 4 personas to test
test_personas = list(persona_system_prompts.keys())[:4]

print(f"INFERENCE TEST - {len(test_personas)} PERSONAS (thinking={'ON' if ENABLE_THINKING else 'OFF'})\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    # enable_thinking must be passed explicitly. Omitting it is NOT the same as False:
    # the template defaults to thinking ON, which prepends Qwen3.8's xhigh reasoning
    # instruction to the persona system prompt and opens an unclosed <think> block -
    # neither of which training ever saw.
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )
    # Passed by keyword: harmless for a tokenizer, and required if this ever runs
    # against a Processor, whose first positional param is `images`, not `text`.
    inputs = tokenizer(text=text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  PERSONA: {persona_key.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024 if ENABLE_THINKING else 256,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        top_k=GEN_TOP_K,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

## 12. Verify Adapter (Reload from Disk)

Final validation: load the adapter cold from disk to confirm it is self-contained and portable.

In [ ]:
# Clean up training model
import gc, torch
del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

# Reload from disk
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

if hasattr(tokenizer2, "tokenizer"):
    tokenizer2 = tokenizer2.tokenizer

# Reload saved system prompts
import json
from pathlib import Path
with open(f"{LORA_OUTPUT_DIR}/persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

# Test with first persona
test_key = list(reloaded_prompts.keys())[0]
test_prompt_text = reloaded_prompts[test_key]

messages = [
    {"role": "system", "content": test_prompt_text},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)
inputs = tokenizer2(text=text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=1024 if ENABLE_THINKING else 256,
    temperature=GEN_TEMPERATURE,
    top_p=GEN_TOP_P,
    top_k=GEN_TOP_K,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (persona: {test_key}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\nAdapter loads cleanly from disk. Ready for deployment via vLLM.")

# List adapter files
print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

## 13. Export Merged Model to GGUF (Mobile / Cross-Platform) — optional

Merge the LoRA adapter into the base model and export to GGUF so it can run anywhere
llama.cpp / Ollama / LM Studio / iOS GGUF runners are supported (e.g. iPhone via apps
like LLMFarm, PocketPal, Private LLM).

- `q4_k_m` is the recommended quant for phones (best size/quality tradeoff).
- `q5_k_m` and `q8_0` are also produced for higher-quality desktop use.
- Requires Unsloth's bundled `llama.cpp` build (downloaded automatically on first call
  to `save_pretrained_gguf`).

**Not needed for the vLLM deployment path** — the LoRA adapter saved in Cell 10 is the
shipping artifact. Skip these cells unless you specifically want GGUF.

**Memory:** this reloads the base at `load_in_4bit=False`, i.e. ~56 GB of bf16 weights,
then merges. Run it in a fresh kernel if training left the pool fragmented.

In [ ]:
# Compatibility shim for PEFT + GPTQModel during GGUF export.
# PEFT imports GPTQModel's older AWQ class name even when this adapter is not AWQ.
try:
    import gptqmodel.nn_modules.qlinear.gemm_awq as _gemm_awq
    if not hasattr(_gemm_awq, "AwqGEMMQuantLinear") and hasattr(_gemm_awq, "AwqGEMMLinear"):
        _gemm_awq.AwqGEMMQuantLinear = _gemm_awq.AwqGEMMLinear
        print("Patched GPTQModel AWQ class alias for PEFT compatibility.")
except Exception as exc:
    print(f"GPTQModel AWQ compatibility shim skipped: {exc}")

In [ ]:
# Reload adapter fresh so we export from a clean state
import gc, torch
from pathlib import Path
from unsloth import FastLanguageModel

try:
    del model2, tokenizer2
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

GGUF_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/gguf"
Path(GGUF_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Load base + adapter; Unsloth's GGUF exporter merges before conversion
model_gguf, tokenizer_gguf = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # full-precision merge for accurate GGUF quantization
)

# Force the chat-template's end-of-turn token as the GGUF EOS.
#
# This is REQUIRED for Qwen3.8, not defensive boilerplate. Verified on the base repo:
#   config.text_config.eos_token_id = 248044  ->  <|endoftext|>
#   chat template closes every turn with     ->  <|im_end|> (248046)
# llama.cpp's converter reads eos_token_id from the nested text_config on
# Conditional-Generation models, so without this it bakes in <|endoftext|>, the model
# never stops on <|im_end|>, and it runs on into the next turn header. iOS GGUF runners
# then strip the leading <|im_start|> special token and render the bare role text
# ("user\n...") plus a hallucinated user question.
_render = tokenizer_gguf.apply_chat_template(
    [{"role": "user", "content": "x"}, {"role": "assistant", "content": "y"}],
    tokenize=False,
)
_eot_candidates = ["<|im_end|>", "<turn|>", "<end_of_turn>", "<|eot_id|>"]
eot_token = next((c for c in _eot_candidates if c in _render), None)
if eot_token is None:
    raise RuntimeError(f"Could not detect end-of-turn marker in chat template. Rendered: {_render!r}")
# The Processor wraps the tokenizer; unwrap so we can call tokenizer methods.
_tok = getattr(tokenizer_gguf, "tokenizer", tokenizer_gguf)
eot_id = _tok.convert_tokens_to_ids(eot_token)
if eot_id is None or eot_id == _tok.unk_token_id:
    raise RuntimeError(f"{eot_token!r} not in tokenizer vocab (got id={eot_id})")

_tok.eos_token = eot_token
model_gguf.config.eos_token_id = eot_id
# Qwen3.8 is a Conditional-Generation model with a nested text_config; the llama.cpp
# converter reads from there, so we MUST set it here too.
if hasattr(model_gguf.config, "text_config") and model_gguf.config.text_config is not None:
    model_gguf.config.text_config.eos_token_id = eot_id
# generation_config.eos_token_id is a list ([248046, 248044]) on this model; collapse to
# the single chat-template EOS so runners that only honor a scalar pick the right one.
if getattr(model_gguf, "generation_config", None) is not None:
    model_gguf.generation_config.eos_token_id = eot_id

print(f"Forced GGUF EOS to {eot_token!r} (id={eot_id})")

# Pass ALL quant methods in one call so the LoRA->FP16 merge happens ONCE
# and llama.cpp quantizes from that single merged file. Otherwise the loop
# re-merges and re-writes the FP16 GGUF for every quant level.
QUANT_METHODS = ["q4_k_m", "q5_k_m", "q8_0"]

print(f"Exporting GGUF (single merge -> {len(QUANT_METHODS)} quants)...")
model_gguf.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer_gguf,
    quantization_method=QUANT_METHODS,
)

print(f"\nGGUF export complete: {GGUF_OUTPUT_DIR}")
for f in sorted(Path(GGUF_OUTPUT_DIR).glob("*.gguf")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:60s} {size_mb:>8.1f} MB")

print("\nMobile usage (iPhone):")
print("  1. Transfer the q4_k_m .gguf file to your phone (AirDrop / Files app).")
print("  2. Open in an iOS GGUF runner (LLMFarm, PocketPal, Private LLM, etc.).")
print("  3. Use Qwen ChatML template; set context length <= MAX_SEQ_LENGTH.")

del model_gguf, tokenizer_gguf
gc.collect()
torch.cuda.empty_cache()